# RAG, SuSteelAible

## problem solving

In [ ]:
# # Delete vector database
# import os
# import shutil
# db_path = "../vector_db/reports"
# if os.path.exists(db_path):
#     shutil.rmtree(db_path)
#     print("✓ Deleted old database")

In [1]:
# # 2. Delete output tables
# if os.path.exists("decarbonisation_tables"):
#     shutil.rmtree("decarbonisation_tables")
#     print("✓ Deleted decarbonisation_tables folder")

## LOADING AND SETTING UP

In [ ]:
# Import necessary libraries

import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy
from transformers import AutoTokenizer
from tqdm import tqdm
from langchain_groq import ChatGroq
import fitz  # PyMuPDF
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain import hub
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

load_dotenv()

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Stop the fork warning

In [21]:
# instantiate a chatgroq model
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

In [23]:
def load_pdfs_with_metadata(base_folder="../data/reports"):
    documents = []

    # Iterate through company folders
    for company_folder in os.listdir(base_folder):
        company_path = os.path.join(base_folder, company_folder)

        # Skip if not a directory
        if not os.path.isdir(company_path):
            continue

        company_name = company_folder

        # Iterate through PDF files in company folder
        for filename in os.listdir(company_path):
            if not filename.lower().endswith(".pdf"):
                continue

            full_path = os.path.join(company_path, filename)
            print(f"Loading: {full_path}")

            try:
                doc = fitz.open(full_path)

                # Extract text from all pages
                text = ""
                for page_num in range(len(doc)):
                    page = doc[page_num]
                    text += page.get_text()

                doc.close()

            except Exception as e:
                print(f"⚠️ Skipping {full_path} because of error: {e}")
                continue

            # Parse metadata from filename
            name, _ = os.path.splitext(filename)
            parts = name.split("_")
            company_id = parts[0] if len(parts) > 0 else None
            year = parts[1] if len(parts) > 1 else None
            report_type = "_".join(parts[2:]) if len(parts) > 2 else None

            # Create document object (adjust to match your Document class)
            doc_obj = {
                'page_content': text,
                'metadata': {
                    'company_name': company_name,
                    'company_id': company_id,
                    'year': year,
                    'report_type': report_type,
                    'filename': filename
                }
            }

            documents.append(doc_obj)

    return documents

In [25]:
# Chunk documents
def chunk_documents(documents, chunk_size=1500, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = text_splitter.split_documents(documents=documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata["id"] = f"chunk_{i}"

    return chunks


todo
- save global vars once and reuse like:
model_name
max_tokens
base_path for vector_db

In [ ]:
# --- Global config (load once) ---
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
MAX_TOKENS = 512

embedding_model = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text: str) -> int:
    """Return number of tokens for a given text."""
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
    )
    return len(encoded["input_ids"])

def embed_and_store_incremental(chunks, db_name: str):
    """
    Add new chunks to existing vectorstore or create new one.
    This does NOT overwrite - it adds to existing database.
    """
    base_path = "vector_db"
    save_path = os.path.join(base_path, db_name)

    # 1) Prepare texts (truncate if needed) and collect metadatas
    texts = []
    metadatas = []
    print("Checking token lengths (and truncating if needed)...")

    for i, chunk in enumerate(chunks):
        text = chunk.page_content
        n_tokens = count_tokens(text)

        if n_tokens > MAX_TOKENS:
            # Truncate to MAX_TOKENS tokens
            encoded = tokenizer(
                text,
                add_special_tokens=False,
                truncation=False,
            )
            truncated_ids = encoded["input_ids"][:MAX_TOKENS]
            text = tokenizer.decode(truncated_ids, skip_special_tokens=True)
            print(
                f"⚠️ Chunk {i} had {n_tokens} tokens; truncated to {MAX_TOKENS} tokens."
            )

        texts.append(text)
        metadatas.append(getattr(chunk, "metadata", {}))

    # 2) Try to load existing vectorstore
    try:
        existing_vectorstore = FAISS.load_local(
            folder_path=save_path,
            embeddings=embedding_model,
            allow_dangerous_deserialization=True,
            distance_strategy=DistanceStrategy.COSINE,
        )
        print(f"✓ Loaded existing vectorstore. Adding {len(texts)} new chunks...")

        # Add new documents to existing store
        existing_vectorstore.add_texts(texts=texts, metadatas=metadatas)
        vectorstore = existing_vectorstore

    except Exception as e:
        print(f"No existing vectorstore found. Creating new one...")
        print(f"(Error was: {e})")

        # Create new vectorstore
        embeddings_list = []
        print("\nEmbedding chunks...")
        for text in tqdm(texts, desc="Embedding", unit="chunk"):
            vec = embedding_model.embed_documents([text])[0]
            embeddings_list.append(vec)

        text_embedding_pairs = list(zip(texts, embeddings_list))
        vectorstore = FAISS.from_embeddings(
            text_embeddings=text_embedding_pairs,
            embedding=embedding_model,
            metadatas=metadatas,
            distance_strategy=DistanceStrategy.COSINE,
        )

    # 3) Save to disk
    os.makedirs(base_path, exist_ok=True)
    vectorstore.save_local(save_path)
    print(f"✓ Saved to {save_path}")

    return vectorstore

def load_report_vectorstore(db_name="reports"):
    base_path = "vector_db"
    vector_db_path = os.path.join(base_path, db_name)

    embeddings = HuggingFaceEmbeddings(
        model_name=MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},
    )

    report_vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE,
    )

    retriever = report_vectorstore.as_retriever()
    return retriever, report_vectorstore


In [ ]:
# function to create retrieval and document processing chains
def connect_chains(retriever):
    stuff_documents_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=hub.pull("langchain-ai/retrieval-qa-chat")
    )

    retrieval_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=stuff_documents_chain
    )

    return retrieval_chain

## MAIN EXECUTION SECTION

In [ ]:
# STEP 1: Load all PDFs from reports folder

print("\n" + "="*50)
print("STEP 1: Loading PDFs")
print("="*50)
reports = load_pdfs_with_metadata("../data/reports")
print(f"✓ Loaded {len(reports)} document pages total")

In [ ]:
# STEP 2: Chunk the documents

print("\n" + "="*50)
print("STEP 2: Chunking documents")
print("="*50)
report_chunks = chunk_documents(reports)
print(f"✓ Created {len(report_chunks)} chunks")

In [14]:
# Check which companies I have

companies = sorted(set(c.metadata.get("company_id") for c in report_chunks))
print(f"✓ Companies found: {companies}")

✓ Companies found: ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '012', '013', '014', '015', '016']


In [ ]:
# STEP 3: Embed and store (or add to existing)

print("\n" + "="*50)
print("STEP 3: Embedding and storing")
print("="*50)
report_vectorstore = embed_and_store_incremental(
    chunks=report_chunks,
    db_name="reports",
)


In [16]:
# STEP 4: Load retriever

print("\n" + "="*50)
print("STEP 4: Loading retriever")
print("="*50)
retriever, report_vectorstore = load_report_vectorstore()
print("✓ Retriever loaded")


STEP 4: Loading retriever
✓ Retriever loaded


In [17]:
# STEP 5: Connect chains

print("\n" + "="*50)
print("STEP 5: Connecting chains")
print("="*50)
report_retrieval_chain = connect_chains(retriever)
print("✓ Chains connected")


STEP 5: Connecting chains
✓ Chains connected


## EXTRACTION FUNCTIONS

In [ ]:
from tabulate import tabulate # pretty-print tables
import pandas as pd
from langchain.prompts import ChatPromptTemplate


In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false" # check if speed concern

def get_years_for_company(chunks, company_id: str):
    """Return sorted list of years for which this company has report chunks."""
    years = sorted(
        {
            c.metadata.get("year")
            for c in chunks
            if c.metadata.get("company_id") == company_id
        }
    )
    return years

In [ ]:
# Prompts & chains for barriers & motivators

barrier_prompt = ChatPromptTemplate.from_template(
    """
You are extracting *barriers to decarbonisation* mentioned in company reports.
Given the context below, output ONLY a list of barriers as short bullet points.

RULES:
- Each barrier MUST be on its own line.
- Each line MUST start with "- ".
- Do NOT write any introductions, explanations, summaries, or extra text.
- If there are no clear barriers, output exactly: "NO_BARRIERS_FOUND".

Context:
{context}

Answer:
"""
)
barrier_chain = barrier_prompt | llm

motivator_prompt = ChatPromptTemplate.from_template(
    """
You are extracting *motivators to decarbonise* mentioned in company reports.
Given the context below, output ONLY a list of motivators as short bullet points.

RULES:
- Each motivator MUST be on its own line.
- Each line MUST start with "- ".
- Do NOT write any introductions, explanations, summaries, or extra text.
- If there are no clear motivators, output exactly: "NO_MOTIVATORS_FOUND".

Context:
{context}

Answer:
"""
)
motivator_chain = motivator_prompt | llm

def extract_list_for_year(
    company_id: str,
    year: str,
    retriever,
    chain,
    query: str,
    k: int = 10,
) -> str:
    """
    Use retriever + LLM chain to extract a bullet list (barriers or motivators)
    for a given company and year.
    """
    docs = retriever.invoke(
        query,
        filter={"company_id": company_id, "year": year},
        search_kwargs={"k": k},
    )

    if not docs:
        return "NO_REPORT_FOR_YEAR"

    context = "\n\n".join(d.page_content for d in docs)
    response = chain.invoke({"context": context})
    text = response.content if hasattr(response, "content") else str(response)

    return text

def clean_bullet_list_text(text: str, empty_token: str = "NO_ITEMS_FOUND") -> str:
    """
    Cleans LLM-generated bullet lists.
    - Returns empty_token if no usable bullets are found.
    - Accepts bullets starting with -, •, *.
    - Removes extra whitespace.
    - Normalizes to one-item-per-line text without leading bullet chars.
    """
    if not text or not isinstance(text, str):
        return empty_token

    text = text.strip()

    # If the model already returned a special token
    if text.upper() == empty_token.upper():
        return empty_token

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    cleaned = []

    for line in lines:
        if line.startswith(("-", "•", "*")):
            item = line.lstrip("-•* ").strip()
            if item:
                cleaned.append(item)

    if not cleaned:
        return empty_token

    return "\n".join(cleaned)

def extract_company_data(company_id: str, report_chunks, retriever):
    """
    Extract barriers and motivators for a specific company across all years.
    Returns two DataFrames: one for barriers, one for motivators.
    """
    print(f"\n{'='*50}")
    print(f"Extracting data for Company {company_id}")
    print(f"{'='*50}")

    years = get_years_for_company(report_chunks, company_id)
    print(f"Years found: {years}")

    barrier_rows = []
    motivator_rows = []

    for year in years:
        print(f"\nProcessing year {year}...")

        # Barriers
        raw_barriers = extract_list_for_year(
            company_id=company_id,
            year=year,
            retriever=retriever,
            chain=barrier_chain,
            query="What barriers to decarbonisation does the company mention?",
            k=10,
        )
        cleaned_barriers = clean_bullet_list_text(
            raw_barriers, empty_token="NO_BARRIERS_FOUND"
        )
        barrier_rows.append(
            {
                "company_id": company_id,
                "year": year,
                "barriers": cleaned_barriers,
            }
        )

        # Motivators
        raw_motivators = extract_list_for_year(
            company_id=company_id,
            year=year,
            retriever=retriever,
            chain=motivator_chain,
            query="What motivators to decarbonise does the company mention?",
            k=10,
        )
        cleaned_motivators = clean_bullet_list_text(
            raw_motivators, empty_token="NO_MOTIVATORS_FOUND"
        )
        motivator_rows.append(
            {
                "company_id": company_id,
                "year": year,
                "motivators": cleaned_motivators,
            }
        )

    df_barriers = pd.DataFrame(barrier_rows)
    df_motivators = pd.DataFrame(motivator_rows)

    return df_barriers, df_motivators

def save_company_tables(company_id: str, df_barriers, df_motivators):
    """Save barriers and motivators tables to CSV and Excel."""
    output_dir = "decarbonisation_tables"
    os.makedirs(output_dir, exist_ok=True)

    # File paths
    barrier_csv_path = os.path.join(output_dir, f"barriers_company_{company_id}.csv")
    barrier_excel_path = os.path.join(output_dir, f"barriers_company_{company_id}.xlsx")
    motivator_csv_path = os.path.join(output_dir, f"motivators_company_{company_id}.csv")
    motivator_excel_path = os.path.join(output_dir, f"motivators_company_{company_id}.xlsx")

    # Save
    df_barriers.to_csv(barrier_csv_path, index=False)
    df_barriers.to_excel(barrier_excel_path, index=False)
    df_motivators.to_csv(motivator_csv_path, index=False)
    df_motivators.to_excel(motivator_excel_path, index=False)

    print(f"\n✓ Saved to {output_dir}/")

    # Display tables
    print("\n" + "="*50)
    print(f"BARRIERS - Company {company_id}")
    print("="*50)
    print(tabulate(df_barriers, headers='keys', tablefmt='grid'))

    print("\n" + "="*50)
    print(f"MOTIVATORS - Company {company_id}")
    print("="*50)
    print(tabulate(df_motivators, headers='keys', tablefmt='grid'))


## USAGE

In [35]:
df_barriers_016, df_motivators_016 = extract_company_data("016", report_chunks, retriever)
save_company_tables("016", df_barriers_016, df_motivators_016)


Extracting data for Company 016
Years found: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

Processing year 2013...

Processing year 2014...

Processing year 2015...

Processing year 2016...

Processing year 2017...

Processing year 2018...

Processing year 2019...

Processing year 2020...

Processing year 2021...

Processing year 2022...

Processing year 2023...

Processing year 2024...

✓ Saved to decarbonisation_tables/

BARRIERS - Company 016
+----+--------------+--------+-------------------+
|    |   company_id |   year | barriers          |
+====+==============+========+===================+
|  0 |          016 |   2013 | NO_BARRIERS_FOUND |
+----+--------------+--------+-------------------+
|  1 |          016 |   2014 | NO_BARRIERS_FOUND |
+----+--------------+--------+-------------------+
|  2 |          016 |   2015 | NO_BARRIERS_FOUND |
+----+--------------+--------+-------------------+
|  3 |          016 |   2016 | NO_BARR

## FIXING ISSUES

In [36]:
# Test if retrieval is working
company_id = "012"  # or whichever company you're testing
year = "2025"  # adjust to a year you know exists

# Test retrieval
docs = retriever.invoke(
    "What barriers to decarbonisation does the company mention?",
    filter={"company_id": company_id, "year": year},
    search_kwargs={"k": 10}
)

print(f"Found {len(docs)} documents")
if docs:
    print("\nFirst document content (first 500 chars):")
    print(docs[0].page_content[:500])
    print("\nFirst document metadata:")
    print(docs[0].metadata)
else:
    print("NO DOCUMENTS FOUND!")

Found 0 documents
NO DOCUMENTS FOUND!


In [37]:
# Get a sample of documents from the vectorstore
all_docs = report_vectorstore.similarity_search("", k=100)

# Check what companies exist
companies = sorted(set(d.metadata.get("company_id") for d in all_docs if d.metadata.get("company_id")))
print(f"Companies in database: {companies}")

# Check what years exist
years = sorted(set(d.metadata.get("year") for d in all_docs if d.metadata.get("year")))
print(f"Years in database: {years}")

# Check specifically for company 012
docs_012 = [d for d in all_docs if d.metadata.get("company_id") == "012"]
print(f"\nCompany 012 documents found: {len(docs_012)}")
if docs_012:
    years_012 = sorted(set(d.metadata.get("year") for d in docs_012))
    print(f"Company 012 years: {years_012}")
    print(f"Sample metadata: {docs_012[0].metadata}")

Companies in database: ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '012', '013', '014', '015', '016']
Years in database: ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

Company 012 documents found: 6
Company 012 years: ['2015', '2017', '2022']
Sample metadata: {'producer': 'PyPDF2 Version 2.12.1', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-02-15T12:28:00+01:00', 'keywords': 'Allgemeine Publikation', 'moddate': '2023-03-06T10:04:43+01:00', 'trapped': 'False', 'author': '', 'title': 'Sustainability Report 2022', 'source': 'reports/012_2022_sustainability_report.pdf', 'total_pages': 79, 'page': 68, 'page_label': '69', 'company_id': '012', 'year': '2022', 'report_type': 'sustainability_report', 'filename': '012_2022_sustainability_report.pdf', 'id': 'chunk_47718'}


In [38]:
import os

# List all PDFs in reports folder
pdf_files = [f for f in os.listdir("reports") if f.endswith(".pdf")]
print(f"PDF files in reports folder: {len(pdf_files)}")

# Show files for company 012
files_012 = [f for f in pdf_files if f.startswith("012")]
print(f"\nCompany 012 files:")
for f in files_012:
    print(f"  {f}")

PDF files in reports folder: 180

Company 012 files:
  012_2019_annual_report.pdf
  012_2020_annual_report.pdf
  012_2019_sustainability_report.pdf
  012_2022_sustainability_report.pdf
  012_2021_annual_report.pdf
  012_2025_sustainability_report.pdf
  012_2015_annual_report.pdf
  012_2024_annual_report.pdf
  012_2022_annual_report.pdf
  012_2016_annual_report.pdf
  012_2023_annual_report.pdf
  012_2017_annual_report.pdf


In [39]:
# Check if the file is in the reports folder
import os
files = os.listdir("reports")
file_012_2025 = [f for f in files if "012" in f and "2025" in f]
print(f"2025 files for company 012 in folder: {file_012_2025}")

# Check what was loaded
print(f"\nTotal report pages loaded: {len(reports)}")
print(f"Total chunks created: {len(report_chunks)}")

# Check if 012_2025 was in the original load
chunks_012_2025 = [c for c in report_chunks if c.metadata.get("company_id") == "012" and c.metadata.get("year") == "2025"]
print(f"\nChunks for 012_2025 in memory: {len(chunks_012_2025)}")

2025 files for company 012 in folder: ['012_2025_sustainability_report.pdf']

Total report pages loaded: 24506
Total chunks created: 68803

Chunks for 012_2025 in memory: 132


In [ ]:
# COMPLETE WORKING SCRIPT - Run this entire cell

import os
import pandas as pd
from langchain.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy
from tabulate import tabulate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Initialize LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

# Load PDFs
def load_pdfs_with_metadata(base_folder="reports"):
    documents = []
    for filename in os.listdir(base_folder):
        if not filename.lower().endswith(".pdf"):
            continue
        full_path = os.path.join(base_folder, filename)

        try:
            loader = PyPDFLoader(full_path)
            docs = loader.load()
        except Exception as e:
            print(f"⚠️ Skipping {full_path}: {e}")
            continue

        name, _ = os.path.splitext(filename)
        parts = name.split("_")
        company_id = parts[0] if len(parts) > 0 else None
        year = parts[1] if len(parts) > 1 else None
        report_type = "_".join(parts[2:]) if len(parts) > 2 else None

        for d in docs:
            d.metadata["company_id"] = company_id
            d.metadata["year"] = year
            d.metadata["report_type"] = report_type
            d.metadata["filename"] = filename

        documents.extend(docs)

    return documents

# Chunk documents
def chunk_documents(documents, chunk_size=1500, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = text_splitter.split_documents(documents=documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata["id"] = f"chunk_{i}"

    return chunks

print("Loading PDFs...")
reports = load_pdfs_with_metadata("reports")
print(f"✓ Loaded {len(reports)} pages")

print("Chunking...")
report_chunks = chunk_documents(reports, chunk_size=1500, chunk_overlap=100)
print(f"✓ Created {len(report_chunks)} chunks")

companies = sorted(set(c.metadata.get("company_id") for c in report_chunks))
print(f"✓ Companies: {companies}")

# Prompts
barrier_prompt = ChatPromptTemplate.from_template(
    """
You are extracting *barriers to decarbonisation* mentioned in company reports.
Given the context below, output ONLY a list of barriers as short bullet points.

RULES:
- Each barrier MUST be on its own line.
- Each line MUST start with "- ".
- Do NOT write any introductions, explanations, summaries, or extra text.
- If there are no clear barriers, output exactly: "NO_BARRIERS_FOUND".

Context:
{context}

Answer:
"""
)
barrier_chain = barrier_prompt | llm

motivator_prompt = ChatPromptTemplate.from_template(
    """
You are extracting *motivators to decarbonise* mentioned in company reports.
Given the context below, output ONLY a list of motivators as short bullet points.

RULES:
- Each motivator MUST be on its own line.
- Each line MUST start with "- ".
- Do NOT write any introductions, explanations, summaries, or extra text.
- If there are no clear motivators, output exactly: "NO_MOTIVATORS_FOUND".

Context:
{context}

Answer:
"""
)
motivator_chain = motivator_prompt | llm

def clean_bullet_list_text(text: str, empty_token: str = "NO_ITEMS_FOUND") -> str:
    if not text or not isinstance(text, str):
        return empty_token

    text = text.strip()

    if text.upper() == empty_token.upper():
        return empty_token

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    cleaned = []

    for line in lines:
        if line.startswith(("-", "•", "*")):
            item = line.lstrip("-•* ").strip()
            if item:
                cleaned.append(item)

    if not cleaned:
        return empty_token

    return "\n".join(cleaned)

def extract_company_data_v2(company_id: str, report_chunks):
    """Extract barriers and motivators directly from chunks in memory."""
    print(f"\n{'='*60}")
    print(f"Extracting data for Company {company_id}")
    print(f"{'='*60}")

    # Get all chunks for this company
    company_chunks = [
        c for c in report_chunks
        if c.metadata.get("company_id") == company_id
    ]

    # Get years
    years = sorted(set(c.metadata.get("year") for c in company_chunks))
    print(f"Years found: {years}")

    barrier_rows = []
    motivator_rows = []

    for year in years:
        print(f"\nProcessing year {year}...")

        # Get chunks for this year
        year_chunks = [c for c in company_chunks if c.metadata.get("year") == year]
        print(f"  → Found {len(year_chunks)} chunks")

        if not year_chunks:
            barrier_rows.append({
                "company_id": company_id,
                "year": year,
                "barriers": "NO_REPORT_FOR_YEAR"
            })
            motivator_rows.append({
                "company_id": company_id,
                "year": year,
                "motivators": "NO_REPORT_FOR_YEAR"
            })
            continue

        # Use first 15 chunks (to avoid context limits)
        context_chunks = year_chunks[:15]
        context = "\n\n".join(c.page_content for c in context_chunks)

        # Extract barriers
        try:
            barrier_response = barrier_chain.invoke({"context": context})
            raw_barriers = barrier_response.content if hasattr(barrier_response, "content") else str(barrier_response)
            cleaned_barriers = clean_bullet_list_text(raw_barriers, empty_token="NO_BARRIERS_FOUND")
        except Exception as e:
            print(f"  ⚠️ Error extracting barriers: {e}")
            cleaned_barriers = "ERROR"

        barrier_rows.append({
            "company_id": company_id,
            "year": year,
            "barriers": cleaned_barriers,
        })

        # Extract motivators
        try:
            motivator_response = motivator_chain.invoke({"context": context})
            raw_motivators = motivator_response.content if hasattr(motivator_response, "content") else str(motivator_response)
            cleaned_motivators = clean_bullet_list_text(raw_motivators, empty_token="NO_MOTIVATORS_FOUND")
        except Exception as e:
            print(f"  ⚠️ Error extracting motivators: {e}")
            cleaned_motivators = "ERROR"

        motivator_rows.append({
            "company_id": company_id,
            "year": year,
            "motivators": cleaned_motivators,
        })

    df_barriers = pd.DataFrame(barrier_rows)
    df_motivators = pd.DataFrame(motivator_rows)

    return df_barriers, df_motivators

def save_company_tables(company_id: str, df_barriers, df_motivators):
    """Save barriers and motivators tables to CSV and Excel."""
    output_dir = "decarbonisation_tables"
    os.makedirs(output_dir, exist_ok=True)

    barrier_csv_path = os.path.join(output_dir, f"barriers_company_{company_id}.csv")
    barrier_excel_path = os.path.join(output_dir, f"barriers_company_{company_id}.xlsx")
    motivator_csv_path = os.path.join(output_dir, f"motivators_company_{company_id}.csv")
    motivator_excel_path = os.path.join(output_dir, f"motivators_company_{company_id}.xlsx")

    df_barriers.to_csv(barrier_csv_path, index=False)
    df_barriers.to_excel(barrier_excel_path, index=False)
    df_motivators.to_csv(motivator_csv_path, index=False)
    df_motivators.to_excel(motivator_excel_path, index=False)

    print(f"\n✓ Saved to {output_dir}/")

    print("\n" + "="*60)
    print(f"BARRIERS - Company {company_id}")
    print("="*60)
    print(tabulate(df_barriers, headers='keys', tablefmt='grid'))

    print("\n" + "="*60)
    print(f"MOTIVATORS - Company {company_id}")
    print("="*60)
    print(tabulate(df_motivators, headers='keys', tablefmt='grid'))

# NOW EXTRACT DATA
print("\n" + "="*70)
print("EXTRACTING DATA FOR ALL COMPANIES")
print("="*70)

for company_id in companies:
    try:
        df_barriers, df_motivators = extract_company_data_v2(company_id, report_chunks)
        save_company_tables(company_id, df_barriers, df_motivators)
        print(f"✓ Completed {company_id}\n")
    except Exception as e:
        print(f"✗ Error with {company_id}: {e}\n")

print("\n" + "="*70)
print("✓ ALL COMPANIES PROCESSED!")
print("="*70)
print(f"Results saved in: decarbonisation_tables/")

In [ ]:
# Make sure you have the retriever loaded
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

def load_report_vectorstore(db_name="reports"):
    base_path = "../vector_db"
    vector_db_path = os.path.join(base_path, db_name)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},  # Fixed typo
    )

    report_vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE,
    )

    retriever = report_vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 50}  # Get more docs for filtering
    )
    return retriever, report_vectorstore

# Load retriever
retriever, report_vectorstore = load_report_vectorstore()

# IMPROVED EXTRACTION FUNCTION
def extract_company_data_v3(company_id: str, report_chunks, retriever):
    """Extract using vector database for relevance."""
    print(f"\n{'='*60}")
    print(f"Extracting data for Company {company_id}")
    print(f"{'='*60}")

    # Get years for this company
    company_chunks = [c for c in report_chunks if c.metadata.get("company_id") == company_id]
    years = sorted(set(c.metadata.get("year") for c in company_chunks))
    print(f"Years found: {years}")

    barrier_rows = []
    motivator_rows = []

    for year in years:
        print(f"\nProcessing year {year}...")

        # BARRIERS - Use vector search to find RELEVANT chunks
        barrier_query = "barriers challenges obstacles difficulties problems constraints to decarbonization carbon reduction emissions reduction climate action"

        all_docs = retriever.invoke(barrier_query, search_kwargs={"k": 50})

        # Filter for this specific company and year
        filtered_docs = [
            d for d in all_docs
            if d.metadata.get("company_id") == company_id
            and d.metadata.get("year") == year
        ]

        print(f"  → Found {len(filtered_docs)} relevant barrier docs")

        if filtered_docs:
            # Use only top 5 most relevant chunks
            context = "\n\n".join(d.page_content for d in filtered_docs[:5])

            try:
                barrier_response = barrier_chain.invoke({"context": context})
                raw_barriers = barrier_response.content if hasattr(barrier_response, "content") else str(barrier_response)
                cleaned_barriers = clean_bullet_list_text(raw_barriers, empty_token="NO_BARRIERS_FOUND")
            except Exception as e:
                print(f"  ⚠️ Error: {e}")
                cleaned_barriers = "ERROR"
        else:
            cleaned_barriers = "NO_BARRIERS_FOUND"

        barrier_rows.append({
            "company_id": company_id,
            "year": year,
            "barriers": cleaned_barriers,
        })

        # MOTIVATORS - Use vector search
        motivator_query = "motivators drivers incentives benefits opportunities reasons goals targets commitments for decarbonization carbon reduction emissions reduction climate action sustainability"

        all_docs = retriever.invoke(motivator_query, search_kwargs={"k": 50})

        filtered_docs = [
            d for d in all_docs
            if d.metadata.get("company_id") == company_id
            and d.metadata.get("year") == year
        ]

        print(f"  → Found {len(filtered_docs)} relevant motivator docs")

        if filtered_docs:
            context = "\n\n".join(d.page_content for d in filtered_docs[:5])

            try:
                motivator_response = motivator_chain.invoke({"context": context})
                raw_motivators = motivator_response.content if hasattr(motivator_response, "content") else str(motivator_response)
                cleaned_motivators = clean_bullet_list_text(raw_motivators, empty_token="NO_MOTIVATORS_FOUND")
            except Exception as e:
                print(f"  ⚠️ Error: {e}")
                cleaned_motivators = "ERROR"
        else:
            cleaned_motivators = "NO_MOTIVATORS_FOUND"

        motivator_rows.append({
            "company_id": company_id,
            "year": year,
            "motivators": cleaned_motivators,
        })

    df_barriers = pd.DataFrame(barrier_rows)
    df_motivators = pd.DataFrame(motivator_rows)

    return df_barriers, df_motivators

# NOW EXTRACT
print("\n" + "="*70)
print("EXTRACTING DATA FOR ALL COMPANIES (WITH RELEVANCE)")
print("="*70)

for company_id in companies:
    try:
        df_barriers, df_motivators = extract_company_data_v3(company_id, report_chunks, retriever)
        save_company_tables(company_id, df_barriers, df_motivators)
        print(f"✓ Completed {company_id}\n")
    except Exception as e:
        print(f"✗ Error with {company_id}: {e}\n")

print("\n✓ ALL DONE!")